# Manual cash portfolio NAV + Sharpe top-12 animation

This standalone notebook ranks the manual cash portfolio test-set results by final NAV and Sharpe, then uses Pillow to draw a top-12 animated growth chart: NAV curves on the left and the current ranking on the right.

In [1]:
from __future__ import annotations

from pathlib import Path
import math

import numpy as np
import pandas as pd
from PIL import Image, ImageColor, ImageDraw, ImageFont


def find_track_b_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "Final_Project" / "track_b",
        cwd.parent if cwd.name == "results" else cwd.parent,
        cwd.parent / "track_b",
    ]
    for candidate in candidates:
        if (candidate / "results" / "manual_cash_portfolio_backtest").exists():
            return candidate
    raise FileNotFoundError("Could not find Final_Project/track_b/results/manual_cash_portfolio_backtest.")


TRACK_B_DIR = find_track_b_dir()
RESULT_DIR = TRACK_B_DIR / "results" / "manual_cash_portfolio_backtest"
OUTPUT_DIR = RESULT_DIR / "top12_nav_sharpe_animation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CA_RANK_PATH = RESULT_DIR / "ca_combined_ranking.csv"
RL_RANK_PATH = RESULT_DIR / "rl_combined_ranking.csv"
NAV_PATH = RESULT_DIR / "combined_holdout_nav.csv"

RANK_TABLE_PATH = OUTPUT_DIR / "test_nav_sharpe_rank.csv"
COMBINED_TOP12_TABLE_PATH = OUTPUT_DIR / "top12_combined_rank.csv"
NAV_TOP12_TABLE_PATH = OUTPUT_DIR / "top12_by_nav_rank.csv"
SHARPE_TOP12_TABLE_PATH = OUTPUT_DIR / "top12_by_sharpe_rank.csv"

COMBINED_FINAL_PNG_PATH = OUTPUT_DIR / "top12_combined_growth_final.png"
COMBINED_GIF_PATH = OUTPUT_DIR / "top12_combined_growth.gif"
NAV_FINAL_PNG_PATH = OUTPUT_DIR / "top12_by_nav_growth_final.png"
NAV_GIF_PATH = OUTPUT_DIR / "top12_by_nav_growth.gif"
SHARPE_FINAL_PNG_PATH = OUTPUT_DIR / "top12_by_sharpe_growth_final.png"
SHARPE_GIF_PATH = OUTPUT_DIR / "top12_by_sharpe_growth.gif"

TOP_N = 12
FRAME_COUNT = 84
FRAME_DURATION_MS = 80
DISPLAY_EXCLUDE_PATTERN = "pca_kmeans_baseline"

print(f"Track B dir: {TRACK_B_DIR}")
print(f"Manual cash result dir: {RESULT_DIR}")

Track B dir: F:\Documents\GitHub\CU_Quant_Trade\Final_Project\track_b
Manual cash result dir: F:\Documents\GitHub\CU_Quant_Trade\Final_Project\track_b\results\manual_cash_portfolio_backtest


In [2]:
def load_rank_table(path: Path, family: str) -> pd.DataFrame:
    frame = pd.read_csv(path)
    frame = frame.copy()
    frame["family"] = family
    frame["model_name"] = frame["experiment_name"].astype(str)
    frame["plot_name"] = family + ":" + frame["model_name"]
    return frame


ca_rank = load_rank_table(CA_RANK_PATH, "CA")
rl_rank = load_rank_table(RL_RANK_PATH, "RL")
ranking_raw = pd.concat([ca_rank, rl_rank], ignore_index=True, sort=False)

nav_abs = pd.read_csv(NAV_PATH, parse_dates=["Date"]).set_index("Date").sort_index()
nav_abs = nav_abs.apply(pd.to_numeric, errors="coerce").ffill().dropna(how="all")

ranking_raw = ranking_raw[ranking_raw["plot_name"].isin(nav_abs.columns)].copy()
if ranking_raw.empty:
    raise ValueError("No ranking rows matched combined_holdout_nav columns.")

curve_final_nav = nav_abs.iloc[-1].rename("curve_final_nav")
ranking_raw = ranking_raw.merge(
    curve_final_nav,
    left_on="plot_name",
    right_index=True,
    how="left",
)
ranking_raw["test_nav"] = ranking_raw["curve_final_nav"].fillna(ranking_raw["test_final_nav"])
ranking_raw["test_sharpe"] = pd.to_numeric(ranking_raw["test_sharpe"], errors="coerce")
ranking_raw["test_nav"] = pd.to_numeric(ranking_raw["test_nav"], errors="coerce")

ranking = ranking_raw.dropna(subset=["test_nav", "test_sharpe"]).copy()
ranking["nav_rank"] = ranking["test_nav"].rank(ascending=False, method="min").astype(int)
ranking["sharpe_rank"] = ranking["test_sharpe"].rank(ascending=False, method="min").astype(int)
ranking["rank_score"] = ranking[["nav_rank", "sharpe_rank"]].mean(axis=1)


def ordered_ranking(rank_frame: pd.DataFrame, mode: str) -> pd.DataFrame:
    sort_specs = {
        "combined": (["rank_score", "nav_rank", "sharpe_rank", "test_nav"], [True, True, True, False]),
        "nav": (["nav_rank", "sharpe_rank", "test_nav"], [True, True, False]),
        "sharpe": (["sharpe_rank", "nav_rank", "test_sharpe"], [True, True, False]),
    }
    sort_columns, ascending = sort_specs[mode]
    ordered = rank_frame.sort_values(sort_columns, ascending=ascending).reset_index(drop=True).copy()
    ordered.insert(0, f"{mode}_position", np.arange(1, len(ordered) + 1))
    return ordered


combined_ranking = ordered_ranking(ranking, "combined").rename(columns={"combined_position": "overall_rank"})
nav_ranking = ordered_ranking(ranking, "nav")
sharpe_ranking = ordered_ranking(ranking, "sharpe")
ranking = combined_ranking.copy()


def remove_display_exclusions(rank_frame: pd.DataFrame) -> pd.DataFrame:
    if not DISPLAY_EXCLUDE_PATTERN:
        return rank_frame.copy()
    keep_mask = ~rank_frame["plot_name"].str.contains(DISPLAY_EXCLUDE_PATTERN, case=False, na=False)
    return rank_frame.loc[keep_mask].copy()


combined_display_ranking = remove_display_exclusions(combined_ranking)
nav_display_ranking = remove_display_exclusions(nav_ranking)
sharpe_display_ranking = remove_display_exclusions(sharpe_ranking)

rank_columns = [
    "overall_rank",
    "nav_position",
    "sharpe_position",
    "family",
    "plot_name",
    "model_name",
    "architecture",
    "test_nav",
    "test_sharpe",
    "nav_rank",
    "sharpe_rank",
    "rank_score",
    "test_total_return",
    "test_annual_return",
    "test_annual_vol",
    "test_max_drawdown",
    "test_calmar",
    "test_total_turnover",
]
combined_columns = [column for column in rank_columns if column in combined_ranking.columns]
ranking_out = combined_ranking.loc[:, combined_columns].copy()
ranking_out.to_csv(RANK_TABLE_PATH, index=False)

top12 = combined_display_ranking.head(TOP_N).copy()
top12_by_nav = nav_display_ranking.head(TOP_N).copy()
top12_by_sharpe = sharpe_display_ranking.head(TOP_N).copy()

top12.loc[:, [column for column in rank_columns if column in top12.columns]].to_csv(COMBINED_TOP12_TABLE_PATH, index=False)
top12_by_nav.loc[:, [column for column in rank_columns if column in top12_by_nav.columns]].to_csv(NAV_TOP12_TABLE_PATH, index=False)
top12_by_sharpe.loc[:, [column for column in rank_columns if column in top12_by_sharpe.columns]].to_csv(SHARPE_TOP12_TABLE_PATH, index=False)

display(ranking_out.head(TOP_N))
print(f"Saved rank table: {RANK_TABLE_PATH}")
print(f"Saved combined top-{TOP_N} table: {COMBINED_TOP12_TABLE_PATH}")
print(f"Saved NAV top-{TOP_N} table: {NAV_TOP12_TABLE_PATH}")
print(f"Saved Sharpe top-{TOP_N} table: {SHARPE_TOP12_TABLE_PATH}")

,overall_rank,family,plot_name,model_name,architecture,test_nav,test_sharpe,nav_rank,sharpe_rank,rank_score,test_total_return,test_annual_return,test_annual_vol,test_max_drawdown,test_calmar,test_total_turnover
0,1,RL,RL:lstm_autoencoder__q_learning,lstm_autoencoder__q_learning,lstm_autoencoder+q_learning,16048.565278,1.206758,1,1,1.0,0.604857,0.162428,0.094895,-0.092584,1.754392,7.2
1,2,RL,RL:kmeans_baseline__q_learning,kmeans_baseline__q_learning,kmeans_baseline+q_learning,15972.945270,1.152410,2,3,2.5,0.597295,0.160682,0.097855,-0.092584,1.735537,2.2
2,3,RL,RL:pca_kmeans_baseline__q_learning,pca_kmeans_baseline__q_learning,pca_kmeans_baseline+q_learning,15972.945270,1.152410,2,3,2.5,0.597295,0.160682,0.097855,-0.092584,1.735537,2.2
3,4,RL,RL:rnn_autoencoder__q_learning,rnn_autoencoder__q_learning,rnn_autoencoder+q_learning,15362.035137,1.181104,6,2,4.0,0.536204,0.146369,0.083360,-0.092507,1.582256,22.4
4,5,RL,RL:hmm_baseline__q_learning,hmm_baseline__q_learning,hmm_baseline+q_learning,15719.956224,1.106823,4,5,4.5,0.571996,0.154801,0.096572,-0.092584,1.672014,10.6
5,6,RL,RL:conv_transformer__q_learning,conv_transformer__q_learning,conv_transformer+q_learning,15385.150004,1.054706,5,7,6.0,0.538515,0.146918,0.093870,-0.095249,1.542459,29.8
6,7,RL,RL:tcn_transformer__q_learning,tcn_transformer__q_learning,tcn_transformer+q_learning,15056.375223,0.947545,7,9,8.0,0.505638,0.139062,0.096195,-0.086421,1.609119,42.4
7,8,CA,CA:tcn_transformer,tcn_transformer,tcn_transformer,14848.985527,0.996778,8,8,8.0,0.484899,0.134046,0.086412,-0.084939,1.578152,41.2
8,9,RL,RL:transformer_autoencoder__q_learning,transformer_autoencoder__q_learning,transformer_autoencoder+q_learning,14265.814999,1.067423,10,6,8.0,0.426581,0.119681,0.067235,-0.084939,1.409028,54.8
9,10,RL,RL:patchtst__q_learning,patchtst__q_learning,patchtst+q_learning,14583.787169,0.649787,9,14,11.5,0.458379,0.127562,0.122577,-0.150058,0.850086,124.4


Saved rank table: F:\Documents\GitHub\CU_Quant_Trade\Final_Project\track_b\results\manual_cash_portfolio_backtest\top12_nav_sharpe_animation\test_nav_sharpe_rank.csv
Saved combined top-12 table: F:\Documents\GitHub\CU_Quant_Trade\Final_Project\track_b\results\manual_cash_portfolio_backtest\top12_nav_sharpe_animation\top12_combined_rank.csv
Saved NAV top-12 table: F:\Documents\GitHub\CU_Quant_Trade\Final_Project\track_b\results\manual_cash_portfolio_backtest\top12_nav_sharpe_animation\top12_by_nav_rank.csv
Saved Sharpe top-12 table: F:\Documents\GitHub\CU_Quant_Trade\Final_Project\track_b\results\manual_cash_portfolio_backtest\top12_nav_sharpe_animation\top12_by_sharpe_rank.csv


In [3]:
top_columns = top12["plot_name"].tolist()
plot_nav = nav_abs.loc[:, top_columns].copy().ffill().dropna(how="all")

# Normalize to each strategy's first test-set value so the chart shows growth on a common scale.
first_values = plot_nav.apply(lambda series: series.dropna().iloc[0] if not series.dropna().empty else np.nan)
plot_nav = plot_nav.divide(first_values, axis=1).dropna(axis=1, how="all").ffill()
top12 = top12[top12["plot_name"].isin(plot_nav.columns)].copy()

if plot_nav.empty or len(plot_nav) < 2:
    raise ValueError("Top-12 NAV frame is empty or too short to animate.")

plot_nav.tail()

,RL:lstm_autoencoder__q_learning,RL:kmeans_baseline__q_learning,RL:rnn_autoencoder__q_learning,RL:hmm_baseline__q_learning,RL:conv_transformer__q_learning,RL:tcn_transformer__q_learning,CA:tcn_transformer,RL:transformer_autoencoder__q_learning,RL:patchtst__q_learning,CA:conv_transformer,CA:cls_token_transformer,CA:encoder_only
Date,,,,,,,,,,,,
2026-04-07,1.570005,1.562607,1.524258,1.537858,1.505104,1.472941,1.452652,1.424051,1.418213,1.371485,1.315447,1.388429
2026-04-08,1.592991,1.585485,1.526671,1.560373,1.527140,1.494506,1.473920,1.426305,1.448462,1.378911,1.343503,1.408757
2026-04-09,1.599258,1.591723,1.530845,1.566512,1.533148,1.500385,1.479719,1.428163,1.453291,1.380992,1.350409,1.414299
2026-04-10,1.597385,1.589858,1.529051,1.564677,1.531352,1.498628,1.477985,1.427308,1.451589,1.378948,1.349126,1.412642
2026-04-13,1.604857,1.597295,1.536204,1.571996,1.538515,1.505638,1.484899,1.426581,1.458379,1.380558,1.359533,1.419250


In [4]:
WIDTH, HEIGHT = 1600, 900
BG = "#f8fafc"
INK = "#0f172a"
MUTED = "#64748b"
GRID = "#d7dee8"
AXIS = "#334155"
PANEL = "#ffffff"
CHART_LEFT, CHART_TOP, CHART_RIGHT, CHART_BOTTOM = 90, 135, 1085, 790
RANK_LEFT, RANK_TOP, RANK_RIGHT, RANK_BOTTOM = 1125, 132, 1555, 792

PALETTE = [
    "#2563eb", "#dc2626", "#16a34a", "#d97706", "#7c3aed", "#0891b2",
    "#db2777", "#65a30d", "#ea580c", "#0f766e", "#9333ea", "#475569",
]


def load_font(size: int, bold: bool = False) -> ImageFont.ImageFont:
    candidates = []
    if bold:
        candidates.extend([
            Path("C:/Windows/Fonts/arialbd.ttf"),
            Path("C:/Windows/Fonts/segoeuib.ttf"),
            Path("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"),
        ])
    else:
        candidates.extend([
            Path("C:/Windows/Fonts/arial.ttf"),
            Path("C:/Windows/Fonts/segoeui.ttf"),
            Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"),
        ])
    for path in candidates:
        if path.exists():
            return ImageFont.truetype(str(path), size=size)
    return ImageFont.load_default()


FONT_TITLE = load_font(32, bold=True)
FONT_SUBTITLE = load_font(20)
FONT_AXIS = load_font(17)
FONT_SMALL = load_font(15)
FONT_RANK = load_font(21, bold=True)
FONT_RANK_SMALL = load_font(16)
FONT_RANK_VALUE = load_font(18, bold=True)


def text_width(draw: ImageDraw.ImageDraw, text: str, font: ImageFont.ImageFont) -> int:
    box = draw.textbbox((0, 0), text, font=font)
    return int(box[2] - box[0])


def fit_text(draw: ImageDraw.ImageDraw, text: str, font: ImageFont.ImageFont, max_width: int) -> str:
    if text_width(draw, text, font) <= max_width:
        return text
    suffix = "..."
    trimmed = text
    while trimmed and text_width(draw, trimmed + suffix, font) > max_width:
        trimmed = trimmed[:-1]
    return trimmed + suffix if trimmed else suffix


def compact_name(name: str) -> str:
    replacements = {
        "encoder_only_market_tuples": "enc_market",
        "transformer_autoencoder": "trans_autoenc",
        "lstm_autoencoder": "lstm_autoenc",
        "rnn_autoencoder": "rnn_autoenc",
        "pca_kmeans_baseline": "pca_kmeans",
        "kmeans_baseline": "kmeans",
        "hmm_baseline": "hmm",
        "cls_token_transformer": "cls_transformer",
        "__q_learning": "+q",
        "_q_learning": "+q",
    }
    output = name
    for old, new in replacements.items():
        output = output.replace(old, new)
    return output


color_map = {
    column: ImageColor.getrgb(PALETTE[i % len(PALETTE)])
    for i, column in enumerate(plot_nav.columns)
}
metric_lookup = top12.set_index("plot_name").to_dict(orient="index")
ranking_description = f"Top {len(plot_nav.columns)} by average NAV rank + Sharpe rank"
live_ranking_subtitle = "live NAV among selected top-12"
footer_text = f"Rank score = average(test NAV rank, test Sharpe rank). Output: {OUTPUT_DIR.name}"
meta_rank_field = "overall_rank"
meta_rank_label = "final rank"
ranking_metric_mode = "nav"

y_min = float(np.nanmin(plot_nav.to_numpy(dtype=float)))
y_max = float(np.nanmax(plot_nav.to_numpy(dtype=float)))
y_pad = max((y_max - y_min) * 0.08, 0.02)
y_min -= y_pad
y_max += y_pad


def x_at(position: int) -> int:
    denominator = max(len(plot_nav) - 1, 1)
    return int(CHART_LEFT + (CHART_RIGHT - CHART_LEFT) * position / denominator)


def y_at(value: float) -> int:
    if not np.isfinite(value):
        value = 1.0
    ratio = (value - y_min) / max(y_max - y_min, 1e-9)
    return int(CHART_BOTTOM - ratio * (CHART_BOTTOM - CHART_TOP))


def rounded_rectangle(draw: ImageDraw.ImageDraw, xy: tuple[int, int, int, int], radius: int, **kwargs) -> None:
    draw.rounded_rectangle(xy, radius=radius, **kwargs)


def current_ranking_values(current: pd.DataFrame) -> pd.Series:
    if ranking_metric_mode != "sharpe":
        return current.iloc[-1].sort_values(ascending=False)

    values = {}
    for column in current.columns:
        series = current[column].pct_change(fill_method=None).fillna(0.0).astype(float)
        risk_free_rate = float(metric_lookup[column].get("test_risk_free_rate", 0.0))
        annual_vol = float(series.std(ddof=0) * math.sqrt(252)) if len(series) >= 2 else np.nan
        if len(series) < 2 or not np.isfinite(annual_vol) or annual_vol <= 0:
            values[column] = -np.inf
        else:
            cumulative = float((1.0 + series).cumprod().iloc[-1])
            annual_return = cumulative ** (252 / len(series)) - 1.0
            values[column] = float((annual_return - risk_free_rate) / annual_vol)
    return pd.Series(values).sort_values(ascending=False)


def format_current_value(value: float) -> str:
    if ranking_metric_mode == "sharpe":
        return "n/a" if not np.isfinite(value) else f"{value:.2f}"
    return f"{value:.3f}x"


def draw_chart_frame(frame_end: int) -> Image.Image:
    frame_end = int(max(2, min(frame_end, len(plot_nav))))
    current = plot_nav.iloc[:frame_end]
    current_date = current.index[-1]

    image = Image.new("RGB", (WIDTH, HEIGHT), BG)
    draw = ImageDraw.Draw(image)

    draw.text((70, 42), "Manual Cash Portfolio Test Set", fill=INK, font=FONT_TITLE)
    subtitle = f"{ranking_description} | Through {current_date.date()}"
    draw.text((72, 85), subtitle, fill=MUTED, font=FONT_SUBTITLE)

    rounded_rectangle(draw, (50, 115, 1108, 820), radius=18, fill=PANEL, outline="#e2e8f0", width=1)
    rounded_rectangle(draw, (1105, 115, 1575, 820), radius=18, fill=PANEL, outline="#e2e8f0", width=1)

    # Grid and y-axis labels.
    y_ticks = np.linspace(y_min, y_max, 6)
    for tick in y_ticks:
        y = y_at(float(tick))
        draw.line((CHART_LEFT, y, CHART_RIGHT, y), fill=GRID, width=1)
        label = f"{tick:.2f}x"
        draw.text((CHART_LEFT - 66, y - 9), label, fill=MUTED, font=FONT_SMALL)

    # X-axis labels.
    x_tick_positions = np.unique(np.linspace(0, len(plot_nav) - 1, 6, dtype=int))
    for pos in x_tick_positions:
        x = x_at(int(pos))
        draw.line((x, CHART_BOTTOM, x, CHART_BOTTOM + 6), fill=AXIS, width=1)
        label = plot_nav.index[int(pos)].strftime("%Y-%m")
        label_width = text_width(draw, label, FONT_SMALL)
        draw.text((x - label_width // 2, CHART_BOTTOM + 14), label, fill=MUTED, font=FONT_SMALL)

    draw.line((CHART_LEFT, CHART_TOP, CHART_LEFT, CHART_BOTTOM), fill=AXIS, width=2)
    draw.line((CHART_LEFT, CHART_BOTTOM, CHART_RIGHT, CHART_BOTTOM), fill=AXIS, width=2)
    draw.text((CHART_LEFT, CHART_TOP - 34), "Normalized NAV", fill=INK, font=FONT_AXIS)

    # NAV curves.
    for column in plot_nav.columns:
        values = current[column].to_numpy(dtype=float)
        points = [(x_at(i), y_at(v)) for i, v in enumerate(values) if np.isfinite(v)]
        if len(points) >= 2:
            draw.line(points, fill=color_map[column], width=4, joint="curve")
        if points:
            x_last, y_last = points[-1]
            draw.ellipse((x_last - 5, y_last - 5, x_last + 5, y_last + 5), fill=color_map[column], outline=PANEL, width=2)

    # Right-side live ranking by the selected metric.
    latest = current_ranking_values(current)
    draw.text((RANK_LEFT, 132), "Current Ranking", fill=INK, font=FONT_RANK)
    draw.text((RANK_LEFT, 162), live_ranking_subtitle, fill=MUTED, font=FONT_SMALL)

    item_top = 202
    item_height = 47
    for rank, (column, value) in enumerate(latest.items(), start=1):
        y = item_top + (rank - 1) * item_height
        color = color_map[column]
        metrics = metric_lookup[column]
        label = compact_name(column)
        label = fit_text(draw, label, FONT_RANK_SMALL, 250)
        draw.rounded_rectangle((RANK_LEFT, y, RANK_RIGHT, y + 36), radius=10, fill="#f8fafc", outline="#e2e8f0")
        draw.text((RANK_LEFT + 12, y + 7), f"{rank:>2}", fill=INK, font=FONT_RANK_VALUE)
        draw.rectangle((RANK_LEFT + 48, y + 10, RANK_LEFT + 62, y + 24), fill=color)
        draw.text((RANK_LEFT + 74, y + 7), label, fill=INK, font=FONT_RANK_SMALL)
        draw.text((RANK_RIGHT - 80, y + 7), format_current_value(float(value)), fill=color, font=FONT_RANK_VALUE)
        selected_rank = int(metrics.get(meta_rank_field, metrics.get("overall_rank", rank)))
        meta = f"{meta_rank_label} {selected_rank} | Sharpe {float(metrics['test_sharpe']):.2f}"
        draw.text((RANK_LEFT + 74, y + 28), meta, fill=MUTED, font=FONT_SMALL)

    draw.text((70, 850), footer_text, fill=MUTED, font=FONT_SMALL)
    return image

In [5]:
def prepare_rank_view(
    selected_ranking: pd.DataFrame,
    description: str,
    live_subtitle: str,
    footer: str,
    rank_field: str,
    rank_label: str,
    metric_mode: str,
) -> None:
    global top12, plot_nav, color_map, metric_lookup, y_min, y_max
    global ranking_description, live_ranking_subtitle, footer_text
    global meta_rank_field, meta_rank_label, ranking_metric_mode

    top12 = selected_ranking.head(TOP_N).copy()
    top_columns = top12["plot_name"].tolist()
    plot_nav = nav_abs.loc[:, top_columns].copy().ffill().dropna(how="all")
    first_values = plot_nav.apply(lambda series: series.dropna().iloc[0] if not series.dropna().empty else np.nan)
    plot_nav = plot_nav.divide(first_values, axis=1).dropna(axis=1, how="all").ffill()
    top12 = top12[top12["plot_name"].isin(plot_nav.columns)].copy()
    if plot_nav.empty or len(plot_nav) < 2:
        raise ValueError("Top-12 NAV frame is empty or too short to animate.")

    color_map = {
        column: ImageColor.getrgb(PALETTE[i % len(PALETTE)])
        for i, column in enumerate(plot_nav.columns)
    }
    metric_lookup = top12.set_index("plot_name").to_dict(orient="index")

    y_min = float(np.nanmin(plot_nav.to_numpy(dtype=float)))
    y_max = float(np.nanmax(plot_nav.to_numpy(dtype=float)))
    y_pad = max((y_max - y_min) * 0.08, 0.02)
    y_min -= y_pad
    y_max += y_pad

    ranking_description = description.format(n=len(plot_nav.columns))
    live_ranking_subtitle = live_subtitle
    footer_text = footer
    meta_rank_field = rank_field
    meta_rank_label = rank_label
    ranking_metric_mode = metric_mode


def save_rank_animation(
    selected_ranking: pd.DataFrame,
    png_path: Path,
    gif_path: Path,
    description: str,
    live_subtitle: str,
    footer: str,
    rank_field: str,
    rank_label: str,
    metric_mode: str,
) -> dict[str, object]:
    prepare_rank_view(
        selected_ranking=selected_ranking,
        description=description,
        live_subtitle=live_subtitle,
        footer=footer,
        rank_field=rank_field,
        rank_label=rank_label,
        metric_mode=metric_mode,
    )

    frame_indices = np.unique(np.linspace(2, len(plot_nav), num=min(FRAME_COUNT, len(plot_nav)), dtype=int))
    frames = [draw_chart_frame(frame_end) for frame_end in frame_indices]

    final_image = draw_chart_frame(len(plot_nav))
    final_image.save(png_path)
    frames[0].save(
        gif_path,
        save_all=True,
        append_images=frames[1:],
        duration=FRAME_DURATION_MS,
        loop=0,
        optimize=True,
    )
    return {
        "png": str(png_path),
        "gif": str(gif_path),
        "frames": len(frames),
        "start_date": plot_nav.index.min().date(),
        "end_date": plot_nav.index.max().date(),
    }


outputs = []
outputs.append(save_rank_animation(
    selected_ranking=combined_display_ranking,
    png_path=COMBINED_FINAL_PNG_PATH,
    gif_path=COMBINED_GIF_PATH,
    description="Top {n} by average NAV rank + Sharpe rank",
    live_subtitle="live ranking by normalized NAV among selected top-12",
    footer="Combined rank score = average(test NAV rank, test Sharpe rank); pca_kmeans excluded from top-12 display.",
    rank_field="overall_rank",
    rank_label="combined rank",
    metric_mode="nav",
))
outputs.append(save_rank_animation(
    selected_ranking=nav_display_ranking,
    png_path=NAV_FINAL_PNG_PATH,
    gif_path=NAV_GIF_PATH,
    description="Top {n} by final test NAV",
    live_subtitle="live ranking by normalized NAV among selected top-12",
    footer="NAV ranking uses final portfolio NAV on the test set; pca_kmeans excluded from top-12 display.",
    rank_field="nav_position",
    rank_label="NAV rank",
    metric_mode="nav",
))
outputs.append(save_rank_animation(
    selected_ranking=sharpe_display_ranking,
    png_path=SHARPE_FINAL_PNG_PATH,
    gif_path=SHARPE_GIF_PATH,
    description="Top {n} by test Sharpe ratio",
    live_subtitle="live ranking by cumulative Sharpe among selected top-12",
    footer="Sharpe ranking uses final test Sharpe; pca_kmeans excluded from top-12 display.",
    rank_field="sharpe_position",
    rank_label="Sharpe rank",
    metric_mode="sharpe",
))

pd.DataFrame(outputs)

,png,gif,frames,start_date,end_date
0,F:\Documents\GitHub\CU_Quant_Trade\Final_Proje...,F:\Documents\GitHub\CU_Quant_Trade\Final_Proje...,84,2023-02-14,2026-04-13
1,F:\Documents\GitHub\CU_Quant_Trade\Final_Proje...,F:\Documents\GitHub\CU_Quant_Trade\Final_Proje...,84,2023-02-14,2026-04-13
2,F:\Documents\GitHub\CU_Quant_Trade\Final_Proje...,F:\Documents\GitHub\CU_Quant_Trade\Final_Proje...,84,2023-02-14,2026-04-13
